### Cell 1: Install Required Libraries


In [1]:
!pip install -q rapidfuzz jellyfish lightgbm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.5/360.5 kB 18.0 MB/s eta 0:00:00


### Cell 2: Setup Paths & Imports


In [ ]:
import os
import sys
import gc
import re
import math
import time
import json
import string
import unicodedata
import subprocess
from collections import defaultdict

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import average_precision_score
from rapidfuzz import fuzz, distance
import jellyfish

# 1. Automatically find dataset path
base_input = "/kaggle/input"
train_path, test_path = None, None

for root, dirs, files in os.walk(base_input):
    if "train_source1.tsv" in files:
        train_path = root
    if "test_source1.tsv" in files:
        test_path = root

if not train_path or not test_path:
    # Check local workspace paths
    for candidate in [
        "student_resource/dataset",
        "../student_resource/dataset",
        "../../../student_resource/dataset"
    ]:
        if os.path.exists(f"{candidate}/train/train_source1.tsv"):
            train_path = f"{candidate}/train"
            test_path = f"{candidate}/test"
            break

if not train_path or not test_path:
    print(f"Could not locate train/test TSV files under {base_input}. Check attached dataset.")
else:
    print(f"[OK] Found Train Data: {train_path}")
    print(f"[OK] Found Test Data:  {test_path}")

# 2. Setup Output Directory
OUTPUT_DIR = "/kaggle/working/output" if os.path.exists("/kaggle") else "code/business_entity_resolution/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"[OK] Output Directory: {OUTPUT_DIR}")


### Cell 3: Text Normalization Module


In [ ]:
SUFFIX_DICT = {
    'corporation': 'corp', 'co': 'corp', 'corp': 'corp',
    'limited': 'ltd', 'ltd': 'ltd', 'ltd.': 'ltd',
    'private': 'pvt', 'pvt': 'pvt',
    'incorporated': 'inc', 'inc': 'inc', 'inc.': 'inc',
    'llc': 'llc', 'l.l.c.': 'llc', 'l.l.c': 'llc',
    'llp': 'llp', 'pllc': 'pllc',
    'pc': 'pc', 'p.c.': 'pc',
    'lp': 'lp',
    'company': 'co'
}

ADDR_ABBR_DICT = {
    'st': 'street', 'st.': 'street',
    'rd': 'road', 'rd.': 'road',
    'ave': 'avenue', 'ave.': 'avenue',
    'apt': 'apartment', 'apt.': 'apartment',
    'blvd': 'boulevard', 'blvd.': 'boulevard',
    'dr': 'drive', 'dr.': 'drive',
    'ln': 'lane', 'ln.': 'lane',
    'ct': 'court', 'ct.': 'court',
    'pl': 'place', 'pl.': 'place',
    'sq': 'square', 'sq.': 'square',
    'hwy': 'highway', 'pkwy': 'parkway'
}

STOPWORDS = {
    'ltd', 'limited', 'inc', 'incorporated', 'corp', 'corporation',
    'pvt', 'private', 'llc', 'the', 'and', 'co', 'company', 'services',
    'enterprises', 'solutions', 'india', 'us', 'usa', 'france', 'group'
}

def clean_text_fast(text):
    if not text or pd.isna(text): return ""
    s = str(text).lower()
    s = re.sub(r'\b&\b', ' and ', s).replace('&', ' and ')
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def normalize_name_fast(text):
    c = clean_text_fast(text)
    tokens = c.split()
    if not tokens:
        return "", ""
    last = tokens[-1]
    if last in SUFFIX_DICT:
        norm_suffix = SUFFIX_DICT[last]
        norm = " ".join(tokens[:-1] + [norm_suffix])
        stripped = " ".join(tokens[:-1])
    else:
        norm = c
        stripped = c
    return norm, stripped

def normalize_addr_fast(text):
    c = clean_text_fast(text)
    tokens = c.split()
    out = [ADDR_ABBR_DICT.get(t, t) for t in tokens]
    norm = " ".join(out)
    
    m = re.findall(r'\b\d{5,6}\b', norm)
    pc = m[-1] if m else ""
    sn_m = re.match(r'^\d+', norm)
    sn = sn_m.group(0) if sn_m else ""
    return norm, pc, sn

def process_dataframe(df):
    df = df.reset_index(drop=True)
    names = df['business_name'].astype(str).tolist() if 'business_name' in df.columns else ['']*len(df)
    addrs = df['business_address'].astype(str).tolist() if 'business_address' in df.columns else ['']*len(df)
    
    name_norm, name_strip = [], []
    for n in names:
        no, st = normalize_name_fast(n)
        name_norm.append(no)
        name_strip.append(st)
        
    addr_norm, postal_code, street_num = [], [], []
    for a in addrs:
        an, pc, sn = normalize_addr_fast(a)
        addr_norm.append(an)
        postal_code.append(pc)
        street_num.append(sn)
        
    df['business_name_raw'] = names
    df['business_name_norm'] = name_norm
    df['business_name_stripped'] = name_strip
    df['business_address_raw'] = addrs
    df['business_address_norm'] = addr_norm
    df['postal_code_guess'] = postal_code
    df['street_number_guess'] = street_num
    return df


### Cell 4: Multi-Strategy Blocking Engine


In [ ]:
class FastBlockingIndex:
    """
    High-performance, memory-bounded blocking engine.
    Uses token inverted indexing, prefix indexing, and rapid candidate
    similarity ranking without memory-heavy TF-IDF matrices or kNN.
    """
    def __init__(self, df_target):
        df_target = df_target.reset_index(drop=True)
        self.target_ids = df_target['entity_id'].values
        self.target_names_norm = df_target['business_name_norm'].values
        self.target_names_strip = df_target['business_name_stripped'].values
        self.target_names_raw = df_target['business_name_raw'].values if 'business_name_raw' in df_target.columns else self.target_names_norm
        self.target_addrs_norm = df_target['business_address_norm'].values
        self.target_countries = df_target['country'].values if 'country' in df_target.columns else ['']*len(df_target)
        self.target_pcs = df_target['postal_code_guess'].values if 'postal_code_guess' in df_target.columns else ['']*len(df_target)
        self.target_sns = df_target['street_number_guess'].values if 'street_number_guess' in df_target.columns else ['']*len(df_target)
        
        n_docs = len(df_target)
        token_freq = defaultdict(int)
        for s in self.target_names_strip:
            for t in set(s.split()):
                if len(t) >= 3 and t not in STOPWORDS:
                    token_freq[t] += 1
                    
        max_freq = max(1000, int(n_docs * 0.02))
        self.inv_index = defaultdict(list)
        self.prefix_index = defaultdict(list)
        
        for idx, s in enumerate(self.target_names_strip):
            for t in set(s.split()):
                if len(t) >= 3 and t not in STOPWORDS and token_freq[t] <= max_freq:
                    self.inv_index[t].append(idx)
            p = s[:4].strip()
            if len(p) >= 3:
                self.prefix_index[p].append(idx)
                
    def query(self, df_source, top_k=20):
        df_source = df_source.reset_index(drop=True)
        s_ids = df_source['entity_id'].values
        s_names_norm = df_source['business_name_norm'].values
        s_names_strip = df_source['business_name_stripped'].values
        
        candidates = []
        for i in range(len(df_source)):
            sid = s_ids[i]
            s_norm = s_names_norm[i]
            s_strip = s_names_strip[i]
            
            cand_indices = set()
            for t in s_strip.split():
                if t in self.inv_index:
                    cand_indices.update(self.inv_index[t])
            p = s_strip[:4].strip()
            if len(p) >= 3 and p in self.prefix_index:
                cand_indices.update(self.prefix_index[p][:100])
                
            if not cand_indices:
                continue
                
            scored = []
            for ci in cand_indices:
                t_norm = self.target_names_norm[ci]
                sim = fuzz.token_sort_ratio(s_norm, t_norm)
                scored.append((sim, ci, self.target_ids[ci]))
                
            scored.sort(reverse=True, key=lambda x: x[0])
            for rank, (sim, ci, teid) in enumerate(scored[:top_k]):
                candidates.append({
                    'source1_entity_id': sid,
                    'candidate_entity_id': teid,
                    'source_idx': i,
                    'target_idx': ci,
                    'block_rank': rank + 1,
                    'score_token': sim / 100.0,
                    'score_tfidf_name': sim / 100.0,
                    'score_tfidf_addr': 0.0,
                    'score_sn': 1.0 if sim >= 80 else 0.0,
                    'score_phon': 0.0
                })
        return pd.DataFrame(candidates)

def run_blocking(df_source, df_target, top_k=20, index=None):
    if index is None:
        print("  -> Building fast token & prefix blocking index...")
        index = FastBlockingIndex(df_target)
    cands = index.query(df_source, top_k=top_k)
    return cands, index


### Cell 5: Feature Engineering & Metrics Module


In [ ]:
def safe_str(s):
    if pd.isna(s): return ""
    return str(s)

def extract_features(df_source, df_target, df_cands, index=None):
    df_source = df_source.reset_index(drop=True)
    
    s_raw = df_source['business_name_raw'].values
    s_norm = df_source['business_name_norm'].values
    s_strip = df_source['business_name_stripped'].values
    s_addr_norm = df_source['business_address_norm'].values
    s_country = df_source['country'].values if 'country' in df_source.columns else ['']*len(df_source)
    s_pc = df_source['postal_code_guess'].values
    s_sn = df_source['street_number_guess'].values
    
    if index is not None:
        t_raw = index.target_names_raw
        t_norm = index.target_names_norm
        t_strip = index.target_names_strip
        t_addr_norm = index.target_addrs_norm
        t_country = index.target_countries
        t_pc = index.target_pcs
        t_sn = index.target_sns
    else:
        df_target = df_target.reset_index(drop=True)
        t_raw = df_target['business_name_raw'].values
        t_norm = df_target['business_name_norm'].values
        t_strip = df_target['business_name_stripped'].values
        t_addr_norm = df_target['business_address_norm'].values
        t_country = df_target['country'].values if 'country' in df_target.columns else ['']*len(df_target)
        t_pc = df_target['postal_code_guess'].values
        t_sn = df_target['street_number_guess'].values
    
    s_indices = df_cands['source_idx'].values
    t_indices = df_cands['target_idx'].values
    b_ranks = df_cands['block_rank'].values
    s_toks = df_cands['score_token'].values
    s_tns = df_cands['score_tfidf_name'].values
    s_sns = df_cands['score_sn'].values
    c_eids = df_cands['candidate_entity_id'].values
    s_eids = df_cands['source1_entity_id'].values
    
    features = []
    for k in range(len(df_cands)):
        si = s_indices[k]
        ti = t_indices[k]
        
        n1_norm = s_norm[si]
        n2_norm = t_norm[ti]
        n1_raw = s_raw[si]
        n2_raw = t_raw[ti]
        
        a1_norm = s_addr_norm[si]
        a2_norm = t_addr_norm[ti]
        
        c1 = str(s_country[si]).lower()
        c2 = str(t_country[ti]).lower()
        t_id = c_eids[k]
        
        f = {
            'source1_entity_id': s_eids[k],
            'candidate_entity_id': t_id,
            'block_rank': b_ranks[k],
            'score_token': s_toks[k],
            'score_tfidf_name': s_tns[k],
            'score_tfidf_addr': 0.0,
            'score_sn': s_sns[k],
            'score_phon': 0.0,
            'country_match': 1 if (c1 == c2 and c1 != '') else 0,
            'source_pair': 1 if 'S2' in t_id else 0,
            'name_lev_raw': distance.Levenshtein.normalized_similarity(n1_raw, n2_raw),
            'name_lev_norm': distance.Levenshtein.normalized_similarity(n1_norm, n2_norm),
            'name_jw': distance.JaroWinkler.similarity(n1_norm, n2_norm)
        }
        
        t1 = set(n1_norm.split())
        t2 = set(n2_norm.split())
        inter = len(t1 & t2)
        union = len(t1 | t2)
        f['name_jaccard'] = inter / union if union > 0 else 0
        f['name_dice'] = 2 * inter / (len(t1) + len(t2)) if len(t1) + len(t2) > 0 else 0
        f['name_token_sort'] = fuzz.token_sort_ratio(n1_norm, n2_norm) / 100.0
        f['name_token_set'] = fuzz.token_set_ratio(n1_norm, n2_norm) / 100.0
        
        l1, l2 = len(n1_norm), len(n2_norm)
        f['name_len_diff_abs'] = abs(l1 - l2)
        f['name_len_diff_rel'] = abs(l1 - l2) / max(l1, l2, 1)
        
        acr1 = "".join([x[0] for x in n1_norm.split() if x])
        acr2 = "".join([x[0] for x in n2_norm.split() if x])
        f['name_acronym_match'] = 1 if ((acr1 == n2_norm.replace(" ", "") or acr2 == n1_norm.replace(" ", "")) and len(acr1) > 1) else 0
        
        n1_st = s_strip[si]
        n2_st = t_strip[ti]
        f['name_exact_strip'] = 1 if (n1_st and n1_st == n2_st) else 0
        
        f['addr_lev_norm'] = distance.Levenshtein.normalized_similarity(a1_norm, a2_norm)
        f['addr_jw'] = distance.JaroWinkler.similarity(a1_norm, a2_norm)
        at1 = set(a1_norm.split())
        at2 = set(a2_norm.split())
        ainter = len(at1 & at2)
        aunion = len(at1 | at2)
        f['addr_jaccard'] = ainter / aunion if aunion > 0 else 0
        
        p1 = s_pc[si]
        p2 = t_pc[ti]
        f['addr_pc_match'] = 1 if (p1 and p2 and p1 == p2) else (-1 if (p1 and p2) else 0)
        
        sn1 = s_sn[si]
        sn2 = t_sn[ti]
        f['addr_sn_match'] = 1 if (sn1 and sn2 and sn1 == sn2) else (-1 if (sn1 and sn2) else 0)
        
        lcs = distance.LCSseq.similarity(a1_norm, a2_norm)
        f['addr_lcs_norm'] = lcs / max(len(a1_norm), len(a2_norm), 1)
        al1, al2 = len(a1_norm), len(a2_norm)
        f['addr_len_diff_rel'] = abs(al1 - al2) / max(al1, al2, 1)
        
        features.append(f)
    return pd.DataFrame(features)

def f0_5_per_entity(predicted_ids: set, true_ids: set) -> float:
    if len(true_ids) == 0:
        return 1.0 if len(predicted_ids) == 0 else 0.0
    if len(predicted_ids) == 0:
        return 0.0
    tp = len(predicted_ids & true_ids)
    precision = tp / len(predicted_ids)
    recall = tp / len(true_ids)
    if precision == 0 and recall == 0:
        return 0.0
    beta = 0.5
    return (1 + beta**2) * precision * recall / (beta**2 * precision + recall)

def macro_f0_5(predictions: dict, ground_truth: dict) -> float:
    scores = [f0_5_per_entity(predictions.get(sid, set()), true_ids)
              for sid, true_ids in ground_truth.items()]
    return sum(scores) / len(scores) if scores else 0.0

def blocking_recall(candidates_df, ground_truth_dict):
    grouped = candidates_df.groupby('source1_entity_id')['candidate_entity_id'].apply(set).to_dict()
    total_true, found_true = 0, 0
    for sid, true_ids in ground_truth_dict.items():
        if not true_ids: continue
        cands = grouped.get(sid, set())
        total_true += len(true_ids)
        found_true += len(true_ids & cands)
    return found_true / total_true if total_true > 0 else 1.0


### Cell 6: Train Model & Optimize F0.5 Threshold


In [ ]:
def load_train_samples(train_path, n_samples=3000, bg_size=5000, random_state=42):
    print(f"Loading {n_samples} training S1 entities and ground truth...")
    df_s1 = pd.read_csv(f"{train_path}/train_source1.tsv", sep='\t', dtype=str).fillna("")
    df_s1 = df_s1.sample(n=n_samples, random_state=random_state).reset_index(drop=True)
    
    df_gt = pd.read_csv(f"{train_path}/train_ground_truth.tsv", sep='\t', dtype=str)
    s1_ids = set(df_s1['entity_id'])
    gt_sub = df_gt[df_gt['source1_entity_id'].isin(s1_ids)]
    
    gt_dict = {}
    targets = set()
    for _, row in gt_sub.iterrows():
        m = str(row['matched_entity_ids'])
        if pd.notna(row['matched_entity_ids']) and m != 'nan' and m:
            t_set = set(m.split(','))
            gt_dict[row['source1_entity_id']] = t_set
            targets.update(t_set)
        else:
            gt_dict[row['source1_entity_id']] = set()
            
    print(f"S1 sample count: {len(df_s1)}, Ground truth targets to retrieve: {len(targets)}")
    
    s2_targets = {t for t in targets if t.startswith('S2-')}
    s3_targets = {t for t in targets if t.startswith('S3-')}
    
    def fetch_from_source(file_path, target_set, bg_n):
        chunks = [pd.read_csv(file_path, sep='\t', dtype=str, nrows=bg_n).fillna("")]
        for chunk in pd.read_csv(file_path, sep='\t', dtype=str, chunksize=250000):
            chunk = chunk.fillna("")
            m = chunk[chunk['entity_id'].isin(target_set)]
            if not m.empty:
                chunks.append(m)
                target_set.difference_update(set(m['entity_id']))
                if not target_set:
                    break
        return pd.concat(chunks).drop_duplicates('entity_id').reset_index(drop=True)
        
    print("Fetching target entities from train_source2 and train_source3...")
    s2_df = fetch_from_source(f"{train_path}/train_source2.tsv", s2_targets.copy(), bg_size)
    s3_df = fetch_from_source(f"{train_path}/train_source3.tsv", s3_targets.copy(), bg_size)
    
    df_target = pd.concat([s2_df, s3_df], ignore_index=True).drop_duplicates('entity_id').reset_index(drop=True)
    print(f"Total target pool: {len(df_target)} entities.")
    return df_s1, df_target, gt_dict

if train_path and test_path:
    print("[1/5] Loading training set...")
    df_train_s1, df_train_s23, gt_dict = load_train_samples(train_path, n_samples=3000, bg_size=5000, random_state=42)
    
    print("[2/5] Normalizing text fields...")
    df_train_s1 = process_dataframe(df_train_s1)
    df_train_s23 = process_dataframe(df_train_s23)
    
    print("[3/5] Splitting Train & Validation (Safe, no in-place shuffle)...")
    np.random.seed(42)
    # IMPORTANT: Use .copy() so in-place shuffle does not scramble DataFrame columns
    s1_ids = df_train_s1['entity_id'].values.copy()
    np.random.shuffle(s1_ids)
    split_idx = int(len(s1_ids) * 0.8)
    train_ids = set(s1_ids[:split_idx])
    val_ids = set(s1_ids[split_idx:])
    
    df_train_split = df_train_s1[df_train_s1['entity_id'].isin(train_ids)].reset_index(drop=True)
    df_val_split = df_train_s1[df_train_s1['entity_id'].isin(val_ids)].reset_index(drop=True)
    
    print("[4/5] Running Multi-Strategy Blocking...")
    df_cands_train, train_index = run_blocking(df_train_split, df_train_s23, top_k=20)
    df_cands_val, _ = run_blocking(df_val_split, df_train_s23, top_k=20, index=train_index)
    
    val_recall = blocking_recall(df_cands_val, {k: gt_dict[k] for k in val_ids})
    print(f"  -> Validation Blocking Recall (top 20): {val_recall:.4f}")
    
    print("[5/5] Extracting features & Training LightGBM...")
    X_train_df = extract_features(df_train_split, df_train_s23, df_cands_train, index=train_index)
    y_train = pd.Series([1 if r['candidate_entity_id'] in gt_dict.get(r['source1_entity_id'], set()) else 0 for _, r in df_cands_train.iterrows()])
    
    X_val_df = extract_features(df_val_split, df_train_s23, df_cands_val, index=train_index)
    y_val = pd.Series([1 if r['candidate_entity_id'] in gt_dict.get(r['source1_entity_id'], set()) else 0 for _, r in df_cands_val.iterrows()])
    
    print(f"  -> Train dataset: {len(X_train_df)} pairs ({y_train.sum()} pos)")
    print(f"  -> Val dataset:   {len(X_val_df)} pairs ({y_val.sum()} pos)")
    
    drop_cols = ['source1_entity_id', 'candidate_entity_id']
    X_train = X_train_df.drop(columns=drop_cols)
    X_val = X_val_df.drop(columns=drop_cols)
    
    model = lgb.LGBMClassifier(
        objective='binary',
        num_leaves=31,
        learning_rate=0.05,
        n_estimators=300,
        random_state=42
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='average_precision',
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
    )
    
    val_preds = model.predict_proba(X_val)[:, 1]
    df_val_res = df_cands_val[['source1_entity_id', 'candidate_entity_id']].copy()
    df_val_res['score'] = val_preds
    
    best_thresh, best_f05 = 0.5, 0.0
    val_gt_subset = {k: gt_dict[k] for k in val_ids}
    for thresh in np.arange(0.15, 0.90, 0.02):
        p_dict = defaultdict(set)
        for _, r in df_val_res[df_val_res['score'] >= thresh].iterrows():
            p_dict[r['source1_entity_id']].add(r['candidate_entity_id'])
        score = macro_f0_5(p_dict, val_gt_subset)
        if score > best_f05:
            best_f05 = score
            best_thresh = thresh
            
    print(f"\n==========================================")
    print(f"Optimal Threshold: {best_thresh:.2f} | Validation F0.5: {best_f05:.4f}")
    print(f"==========================================")
    
    # Clean up all train memory thoroughly
    del df_train_s1, df_train_s23, train_index, df_train_split, df_val_split, gt_dict
    del df_cands_train, df_cands_val, X_train_df, X_val_df, X_train, X_val
    gc.collect()


### Cell 7: Memory-Bounded Chunked Test Inference


In [ ]:
def load_country_targets(test_path, country):
    """
    Loads and normalizes test target sources (S2 & S3) for a single country.
    Guarantees peak memory stays strictly bounded (< 2-4 GB per country).
    """
    print(f"\n--- Loading target sources for country: '{country}' ---")
    t0 = time.time()
    s2_chunks = []
    for chunk in pd.read_csv(f"{test_path}/test_source2.tsv", sep='\t', dtype=str, chunksize=500000):
        m = chunk[chunk['country'] == country]
        if not m.empty:
            s2_chunks.append(m)
    s2_df = pd.concat(s2_chunks, ignore_index=True) if s2_chunks else pd.DataFrame()
    
    s3_chunks = []
    for chunk in pd.read_csv(f"{test_path}/test_source3.tsv", sep='\t', dtype=str, chunksize=500000):
        m = chunk[chunk['country'] == country]
        if not m.empty:
            s3_chunks.append(m)
    s3_df = pd.concat(s3_chunks, ignore_index=True) if s3_chunks else pd.DataFrame()
    
    df_target = pd.concat([s2_df, s3_df], ignore_index=True).drop_duplicates('entity_id').reset_index(drop=True)
    del s2_chunks, s3_chunks, s2_df, s3_df
    gc.collect()
    
    df_target = process_dataframe(df_target)
    print(f"  Loaded & normalized {len(df_target):,} '{country}' targets in {time.time() - t0:.1f}s")
    return df_target

if train_path and test_path:
    print("=== Starting Country-Partitioned Test Inference ===")
    start_infer_time = time.time()
    
    matched_results_map = {}
    candidate_pairs_map = {}
    
    countries = ['France', 'US', 'India']
    
    for country in countries:
        # 1. Load targets only for current country (bounded memory)
        df_country_target = load_country_targets(test_path, country)
        
        # 2. Build fast blocking index for this country
        print(f"  Building blocking index for '{country}'...")
        country_index = FastBlockingIndex(df_country_target)
        
        # 3. Stream test_source1.tsv, process entities matching this country
        print(f"  Processing test_source1 entities for '{country}' in streaming chunks...")
        chunk_size = 25000
        for chunk in pd.read_csv(f"{test_path}/test_source1.tsv", sep='\t', dtype=str, chunksize=chunk_size):
            sub_chunk = chunk[chunk['country'] == country].copy().reset_index(drop=True)
            if sub_chunk.empty:
                continue
                
            sub_chunk = process_dataframe(sub_chunk)
            df_cands_chunk, _ = run_blocking(sub_chunk, df_country_target, top_k=20, index=country_index)
            
            cand_dict = defaultdict(list)
            if not df_cands_chunk.empty:
                for _, r in df_cands_chunk.iterrows():
                    cand_dict[r['source1_entity_id']].append(r['candidate_entity_id'])
                    
                X_test_df = extract_features(sub_chunk, df_country_target, df_cands_chunk, index=country_index)
                drop_cols = ['source1_entity_id', 'candidate_entity_id']
                X_test_features = X_test_df.drop(columns=drop_cols)
                
                test_preds = model.predict_proba(X_test_features)[:, 1]
                df_cands_chunk['score'] = test_preds
                
                pred_dict = defaultdict(list)
                for _, r in df_cands_chunk[df_cands_chunk['score'] >= best_thresh].iterrows():
                    pred_dict[r['source1_entity_id']].append(r['candidate_entity_id'])
            else:
                pred_dict = {}
                
            for sid in sub_chunk['entity_id']:
                matches = list(dict.fromkeys(pred_dict.get(sid, [])))
                cands = list(dict.fromkeys(cand_dict.get(sid, [])))
                matched_results_map[sid] = ",".join(matches) if matches else ""
                candidate_pairs_map[sid] = ",".join(cands) if cands else ""
                
            del sub_chunk, df_cands_chunk
            gc.collect()
            
        del df_country_target, country_index
        gc.collect()
        print(f"  Finished country '{country}'! RAM reset.\n")
        
    # 4. Stream original test_source1.tsv to write exact output order
    print("Writing final submission files in exact test_source1 order...")
    matching_path = os.path.join(OUTPUT_DIR, "matching_results.tsv")
    candidate_path = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
    
    with open(matching_path, 'w', encoding='utf-8') as f_match, open(candidate_path, 'w', encoding='utf-8') as f_cand:
        f_match.write("source1_entity_id\tmatched_entity_ids\n")
        f_cand.write("source1_entity_id\tcandidate_entity_ids\n")
        
        for chunk in pd.read_csv(f"{test_path}/test_source1.tsv", sep='\t', dtype=str, usecols=['entity_id'], chunksize=50000):
            m_lines = []
            c_lines = []
            for eid in chunk['entity_id']:
                m_str = matched_results_map.get(eid, "")
                c_str = candidate_pairs_map.get(eid, "")
                m_lines.append(f"{eid}\t{m_str}\n")
                c_lines.append(f"{eid}\t{c_str}\n")
            f_match.writelines(m_lines)
            f_cand.writelines(c_lines)
            
    print(f"[OK] Successfully generated: {matching_path}")
    print(f"[OK] Successfully generated: {candidate_path}")
    print(f"Total inference time: {(time.time() - start_infer_time)/60:.2f} mins")
    
    # 5. Run validator if present
    val_script = None
    for v_candidate in [
        "student_resource/utils/validate_submission.py",
        "../student_resource/utils/validate_submission.py",
        "/kaggle/working/student_resource/utils/validate_submission.py"
    ]:
        if os.path.exists(v_candidate):
            val_script = v_candidate
            break
            
    if val_script:
        print("Running submission validator...")
        cmd = [sys.executable, val_script,
               '--matching', matching_path,
               '--candidate', candidate_path,
               '--test-dir', test_path]
        res = subprocess.run(cmd, capture_output=True, text=True)
        print(res.stdout)
        if res.returncode != 0:
            print(res.stderr)
    else:
        print("Validator script not found, output files generated successfully.")
